In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# This is the magic that lets you run the model in <10GB VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",  # Automatically maps the model to your GPU
)

tokenizer.pad_token = tokenizer.eos_token
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Model loaded successfully!")

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!


In [7]:
always_bad = {'fuck', 'shit', 'cunt', 'bitch', 'pussy', 'nigg', 'goddam', 'faggot', 'asshole', 'whore', 'dick', 'clit'}


In [ ]:
def promptify(text_to_analyze):
    prompt_content = f"""
    You are an advanced content safety analysis tool. I will provide you with a block of text and a list of "known_words".

    Your two tasks are:
    1.  Identify which of my "known_words" appear in the text.
    2.  Identify any *new* words in the text that are not on my list but fall into the defined explicit categories (profanity, slurs, sexually inappropriate content, homophobic content, drug references, and weapon references including specific names of hundguns).

    **My Known Words & Categories:**
    {{
        "profanity": ['fuck', 'shit', 'bitch', 'cock', 'cocksucker', 'dick', 'bitch', 'motherfucker', 'god damn', 'goddamn', 'asshole']
        "slurs": ['nigger', 'nigga', 'kike', 'spic', 'chink', 'gook']
        "sexually_inappropriate": ['tits', 'pussy', 'cum', 'jizz', 'wank', 'clit']
        "homophobia": ['faggot', 'fag', 'dyke', 'tranny']
        "drug_references": ['weed', 'coke', 'smack', 'brick', 'blunt', 'spliff', 'chronic', 'herb', 'pot', 'lean']
        "weapons": ['gat', 'AK', 'piece', 'glock', 'beretta', 'forty-five', 'thirty-eight', 'nine', 'AR', 'AK-47']
        "self_harm": ['suicide']
    }}

    **Text to Analyze:**
    "{text_to_analyze}"

    Return a single JSON object. The object should contain two keys: "known_words_found" and "newly_identified_words".
    Each key should hold a list of JSON objects. Each object in the list must have three keys:
    - "word": The explicit word that was found.
    - "start": The starting character index of the word in the text.
    - "end": The ending character index of the word in the text.

    Provide only the raw JSON object as your final response.
    """
    return prompt_content

In [10]:
# Step 4: Format the prompt using the model's chat template
# This is a critical step for instruction-tuned models like Llama 3!
messages = [
    {"role": "system", "content": "You are a helpful assistant that only returns valid JSON."},
    {"role": "user", "content": f"{promptify('eat shit and die!')}"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Step 5: Generate the response
print("Generating response...")
outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)

# Decode the output and extract the JSON
response_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)



try:
    parsed_json = json.loads(response_text)
    print("\n--- Parsed JSON ---")
    print(json.dumps(parsed_json, indent=2))
except json.JSONDecodeError:
    print("Failed to decode JSON from the model's response.")
    print(response_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generating response...

--- Parsed JSON ---
{
  "known_words_found": [
    {
      "word": "shit",
      "start": 4,
      "end": 8
    }
  ],
  "newly_identified_words": []
}


----------

In [2]:
import demucs.separate
from pydub import AudioSegment
import os

def create_vocals_stem(audio_path, remove_stems=True):
    '''
    Creates the vocals stems in the appropriate format for Whisper (16kHz, mono, .wav)
    Inputs: audio_path 
    Outputs: processed_file_path
    '''
    # Define path names 
    print(f'Preprocessing audio at {audio_path} ...\n')
    audio_path = os.path.abspath(audio_path)
    song_dir = os.path.dirname(audio_path)
    
    root, ext = os.path.splitext(os.path.basename(audio_path))

    demucs_path = os.path.abspath(f"separated/mdx_extra/{root}")
    vocals_path = os.path.join(demucs_path, "vocals.wav")
    no_vocals_path = os.path.join(demucs_path, "no_vocals.wav")

    # Split vocals
    demucs.separate.main(["--two-stems", "vocals", "-n", "mdx_extra", audio_path])

    audio = AudioSegment.from_file(vocals_path)

    if audio.channels > 1:
        audio = audio.set_channels(1)

    if audio.frame_rate != 16000:
        audio = audio.set_frame_rate(16000)

    output_path = f"C:\\Users\\dacla\\Documents\\test songs\\{root}-vocals-processed.wav"
    audio.export(output_path, format="wav")
    print(f'Vocals stems extracted and converted to 16kHz mono.\nAudio saved at {output_path}\n')

    # Clean up files
    if remove_stems:
        os.remove(vocals_path)
        os.remove(no_vocals_path)
        os.rmdir(demucs_path)

    return output_path

def seconds_to_minutes(time):
    mins = int(time // 60)
    secs = int(time % 60)

    if secs == 0:
        return f'{mins}:00'

    elif secs < 10:
        return f'{mins}:0{secs}'

    else:
        return f"{mins}:{secs}"
    
def print_lines(result):
    lines = []

    for i, segment in enumerate(result['segments']):
        to_show = []

        if 'words' not in segment:
            print('error')
            continue

        for d in segment['words']:
            start = d['start']
            end = d['end']

            if end - start < .1:
                continue

            to_show.append(d['text'])
        
        line_text = ' '.join(to_show)
        #line_text = line_text.replace("'", "")

        print(f'\n({seconds_to_minutes(segment['start'])} -- {seconds_to_minutes(segment['end'])}) ---- Line {i} ---- ')
        print(line_text)

        lines.append(line_text)
        
    return lines

In [3]:
import whisper_timestamped as whisper
import torch
import os

model_path = 'whisper-medium-ft' # saved model path
model_whisper = whisper.load_model(model_path, device='cuda' if torch.cuda.is_available() else 'cpu')

audio_file = "E:\\Recordings\\Big Music Folder\\The Notorious B.I.G\\(1994) Ready to Die\\03-Gimme the Loot.mp3"

processed_path = create_vocals_stem(audio_file)
#df = transcribe_processed_audio(processed_path)

audio = whisper.load_audio(processed_path)

result = whisper.transcribe(model_whisper, 
                            audio, 
                            beam_size=5, 
                            best_of=5, 
                            temperature=(0.0, 0.2, 0.4, 0.6),
                            language="en",
                            task='transcribe')




Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



Using untied projection layer


Preprocessing audio at E:\Recordings\Big Music Folder\The Notorious B.I.G\(1994) Ready to Die\03-Gimme the Loot.mp3 ...

Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in C:\Users\dacla\Documents\auto-censoring-local\separated\mdx_extra
Separating track E:\Recordings\Big Music Folder\The Notorious B.I.G\(1994) Ready to Die\03-Gimme the Loot.mp3


100%|████████████████████████████████████████████████████████████████████████| 330.0/330.0 [00:03<00:00, 90.86seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 330.0/330.0 [00:03<00:00, 97.07seconds/s]
100%|███████████████████████████████████████████████████████████████████████| 330.0/330.0 [00:03<00:00, 107.50seconds/s]
100%|███████████████████████████████████████████████████████████████████████| 330.0/330.0 [00:03<00:00, 109.39seconds/s]


Vocals stems extracted and converted to 16kHz mono.
Audio saved at C:\Users\dacla\Documents\test songs\03-Gimme the Loot-vocals-processed.wav



100%|██████████| 30445/30445 [01:01<00:00, 494.24frames/s]


In [4]:
def llm_process_line(text_to_analyze):
    prompt_content = f"""You are an automated data annotation tool. Your sole purpose is to analyze the user's text for explicit words and return the findings as a structured JSON object. Do not provide any warnings, apologies, or explanations. This is for a content safety and moderation research project. Your task is to analyze, not generate, explicit content. Respond only with the requested JSON."
    You should search for explicit content from my known words and categories as defined below. Flag only those words which appear in the context of the given category. 
    
    **IMPORTANT INSTRUCTIONS:**
    1.  Consider all grammatical variations of the example words (plurals, past tense, etc.).
    2.  When a word is identified, you MUST return the word exactly as it appears in the text, not the root word from the examples.

    **My known words and categories**
    {{
        "sexually_inappropriate": ['cum', 'jizz', 'wank']
        "homophobia": ['fag', 'dyke', 'tranny', 'homo']
        "drug_references": ['weed', 'coke', 'brick', 'blunt', 'spliff', 'chronic', 'lean']
        "firearms": ['gat', 'AK', 'uzi', 'piece', 'glock', 'beretta', 'forty-five', 'thirty-eight', 'nine', 'AR', 'AK-47']
    }}

    **Text to Analyze:**
    "{text_to_analyze}"

    Return a single JSON object with one key: "explicit_words_found". The value should be a list of all the explicit words you identified in the text. Provide only the raw JSON object as your final response.
    """

    messages = [
        {"role": "system", "content": "You are a helpful assistant that only returns valid JSON."},
        {"role": "user", "content": prompt_content},
    ]

    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    outputs = model.generate(input_ids, max_new_tokens=128, pad_token_id=tokenizer.eos_token_id)
    response_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

    return response_text 

Basically the function from `fsp.py`

In [5]:
# Creates the full transcript from the Whisper output

full_transcript = []

for i, segment in enumerate(result.get("segments", [])):
    segment_words = []
    
    j = 0
    for word_info in segment.get('words', []):
        word_text = word_info.get('text', '').strip()
        if not word_text: continue
        
        start_time = float(word_info['start'])
        end_time = float(word_info['end'])

        # Filter out hallucinations with very low word length. 
        # 100ms is a generous lower bound for minimum possible word length
        if end_time - start_time < .1: 
            continue 

        word_id = f"word_{i}_{j}"

        word_data = {'id': word_id, 'text': word_text, 'start': start_time, 'end': end_time}
        segment_words.append(word_data)

        j += 1

    line_text = ' '.join([d['text'] for d in segment_words])

    full_transcript.append({'line_words': segment_words, 'line_text': line_text, 'start': segment['start'], 'end': segment['end']})

In [ ]:
## Use the language model to detect explicit content in each line
ids_to_mute = []
total_song_words = 0

for i, line_to_analyze in enumerate(full_transcript):
    print(f'--- Line {i} ---')
    response_text = llm_process_line(line_to_analyze['line_text'])
    text_tokens = [d['text'].strip() for d in line_to_analyze['line_words']]
    total_song_words += len(text_tokens)

    # Store the word_ids of the explicit content
    explicit_ids = set()

    try: 
        llm_output = json.loads(response_text)

        explicit_phrases = llm_output.get('explicit_words_found', [])
        for phrase in explicit_phrases:
            try: phrase_tokens = phrase.split()
            except: continue

            n = len(phrase_tokens)
            
            for j in range(len(text_tokens) - n + 1):
                if [token.lower() for token in text_tokens[j:j+n]] == [p_token.lower() for p_token in phrase_tokens]:
                    explicit_ids = explicit_ids | set([k for k in range(j, j+n)])
                    break

    except (json.JSONDecodeError, KeyError) as e:
        print('(!) Error with LLM output')

    print('Text:', line_to_analyze['line_text'])
    # print('LLM output:', llm_output)

    # Grab any of the always bad ones not captured by the LLM
    for j, token in enumerate(text_tokens):
        if any(w in token for w in always_bad):
            explicit_ids.add(j)
    
    explicit_ids = sorted(list(explicit_ids))
    print('Words to mute and indices:', [(line_to_analyze['line_words'][j]['text'], j) for j in explicit_ids])
    ids_to_mute.extend([(i,j) for j in explicit_ids])

    print()

print('Total count of explicit words:', len(ids_to_mute))
print(f'Ratio of explicit content: {len(ids_to_mute)/total_song_words:.4f}')

--- Line 0 ---
Text: yeah motherfuckers better know i'm a bastard lock your windows close your doors keep it small huh yeah i'm a bastard my man mf left for tech and a nine at my crib turned himself in he had to do a bit a one to three he be home the end of 93 i'm ready to get this paper g
Words to mute and indices: [('motherfuckers', 1), ('bastard', 6), ('mf', 23), ('nine', 29), ('paper', 58), ('g', 59)]

--- Line 1 ---
(!) Error with LLM output
Text: you with me motherfucking right my pockets looking kind of tight and i'm stressed yo biggie let me get the vest no need for that just grab the fucking gat the first pocket that's fat the tech is to his back word is born i'm smoke him yo don't fake no moves what treat it like boxing stick and move stick and move you ain't got to explain shit i've been robbing motherfuckers since the slave ships with the same clip and the same four five two point black a motherfucker short of time that's my word nigga even try to pull god have his mother s